# Notebook 04 — Explicabilidade com SHAP e LIME**Referência na monografia:** Seção 3.5 (Aplicação das Técnicas de XAI)Este é o **núcleo da contribuição** do trabalho. Aqui as predições deixam de ser números epassam a ser hipóteses de negócio.**Etapas executadas:**1. Análise global com SHAP — *summary plot*, *dependence plot*, importância média (§3.5.1)2. Análise local com LIME — casos selecionados por critério misto (§3.5.2)3. Síntese e tradução em recomendações de negócio (§3.5.3)

In [ ]:
import osimport pickleimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport shapfrom lime.lime_tabular import LimeTabularExplainerwarnings.filterwarnings('ignore')RANDOM_STATE = 42np.random.seed(RANDOM_STATE)BASE_DIR = os.path.dirname(os.getcwd())DATA_DIR = os.path.join(BASE_DIR, 'data')FIG_DIR = os.path.join(BASE_DIR, 'outputs', 'figuras')TAB_DIR = os.path.join(BASE_DIR, 'outputs', 'tabelas')MOD_DIR = os.path.join(BASE_DIR, 'outputs', 'modelos')plt.rcParams['figure.dpi'] = 110plt.rcParams['savefig.dpi'] = 300plt.rcParams['savefig.bbox'] = 'tight'def salvar_fig(nome):    plt.savefig(os.path.join(FIG_DIR, f'{nome}.png'))with open(os.path.join(DATA_DIR, 'dados_processados.pkl'), 'rb') as f:    dados = pickle.load(f)with open(os.path.join(MOD_DIR, 'modelos_treinados.pkl'), 'rb') as f:    art = pickle.load(f)X_teste = dados['X_teste_esc']y_teste = dados['y_teste']X_treino = dados['X_treino_esc']COLUNAS = dados['colunas']modelo = art['modelo_final']modelo_nome = art['modelo_final_nome']limiar = art['limiar_final']prob_teste = art['probabilidades'][modelo_nome]pred_teste = art['predicoes'][modelo_nome]print(f'Modelo em análise: {modelo_nome}')print(f'Limiar de decisão: {limiar:.4f}')print(f'Instâncias no conjunto de teste: {len(X_teste):,}')

## 1. Análise global com SHAP (§3.5.1)A variante do explicador é escolhida conforme o algoritmo selecionado: `TreeExplainer`(TreeSHAP) para modelos baseados em árvores, `LinearExplainer` (LinearSHAP) para aRegressão Logística.

In [ ]:
if modelo_nome in ('Random Forest', 'XGBoost'):    explicador = shap.TreeExplainer(modelo)    variante = 'TreeSHAP'    shap_values = explicador.shap_values(X_teste)    # Random Forest retorna lista por classe; seleciona a classe positiva    if isinstance(shap_values, list):        shap_values = shap_values[1]    elif shap_values.ndim == 3:        shap_values = shap_values[:, :, 1]    valor_base = explicador.expected_value    if isinstance(valor_base, (list, np.ndarray)):        valor_base = np.array(valor_base).ravel()[-1]else:    explicador = shap.LinearExplainer(modelo, X_treino)    variante = 'LinearSHAP'    shap_values = explicador.shap_values(X_teste)    valor_base = explicador.expected_valueprint(f'Variante utilizada: {variante}')print(f'Matriz de valores SHAP: {np.shape(shap_values)}')print(f'Valor esperado (baseline): {float(valor_base):.4f}')

### 1.1 Summary plot — visão global do modelo

In [ ]:
plt.figure(figsize=(10, 8))shap.summary_plot(shap_values, X_teste, feature_names=COLUNAS,                  max_display=15, show=False)plt.title(f'SHAP Summary Plot — {modelo_nome}', fontsize=13, pad=15)plt.xlabel('Valor SHAP (impacto na predição de evasão)')plt.tight_layout()salvar_fig('shap_summary_plot')plt.show()

In [ ]:
# Versão em barras: importância média absolutaplt.figure(figsize=(9, 7))shap.summary_plot(shap_values, X_teste, feature_names=COLUNAS,                  plot_type='bar', max_display=15, show=False, color='#2E86AB')plt.title(f'Importância média das variáveis — {modelo_nome}', fontsize=13, pad=15)plt.xlabel('Valor SHAP médio absoluto')plt.tight_layout()salvar_fig('shap_importancia_barras')plt.show()

### 1.2 Ranking de importância globalCálculo da importância média absoluta, viabilizando a comparação direta com os *rankings*reportados na literatura correlata (Rasool et al., 2025; Lalwani et al., 2022).

In [ ]:
importancia = pd.DataFrame({    'Variável': COLUNAS,    'SHAP médio absoluto': np.abs(shap_values).mean(axis=0),    'SHAP médio (com sinal)': shap_values.mean(axis=0)}).sort_values('SHAP médio absoluto', ascending=False).reset_index(drop=True)importancia['Direção predominante'] = np.where(    importancia['SHAP médio (com sinal)'] > 0,    'Aumenta risco de evasão', 'Reduz risco de evasão')importancia = importancia.round(4)print('RANKING GLOBAL DE IMPORTÂNCIA (15 primeiras)')display(importancia.head(15))importancia.to_csv(os.path.join(TAB_DIR, 'tab_importancia_shap.csv'), index=False)

### 1.3 Dependence plots — variáveis mais relevantesRevelam padrões não-lineares e interações — exatamente o tipo de comportamento que motivoua adoção de modelos de *ensemble* em detrimento de abordagens lineares.

In [ ]:
top_variaveis = importancia['Variável'].head(4).tolist()print(f'Variáveis analisadas: {top_variaveis}')for var in top_variaveis:    plt.figure(figsize=(8, 5))    shap.dependence_plot(var, shap_values, X_teste, feature_names=COLUNAS,                         show=False)    plt.title(f'SHAP Dependence Plot — {var}', fontsize=12, pad=12)    plt.tight_layout()    salvar_fig(f'shap_dependence_{var.replace(" ", "_").replace("/", "_")}')    plt.show()

### 1.4 Confronto com as hipóteses da EDAValidação da coerência entre as explicações produzidas pelo modelo e as hipóteses levantadasna análise exploratória, conforme previsto ao final da Seção 3.5.1.

In [ ]:
caminho_hip = os.path.join(TAB_DIR, 'tab_hipoteses_eda.csv')if os.path.exists(caminho_hip):    hipoteses = pd.read_csv(caminho_hip)    display(hipoteses)print('\nTop 10 variáveis segundo o SHAP:')for i, linha in importancia.head(10).iterrows():    print(f'  {i + 1:>2}. {linha["Variável"]:<35} '          f'{linha["SHAP médio absoluto"]:.4f}  ({linha["Direção predominante"]})')print('\n>> Compare este ranking com as hipóteses H1-H7 e registre no Capítulo 4')print('   quais foram confirmadas, quais foram refutadas e quais surpreenderam.')

## 2. Análise local com LIME (§3.5.2)Seleção de instâncias por **critério misto**, garantindo diversidade dos casos analisados:acertos de alta confiança em ambas as classes, erros do modelo (FP e FN) e casos-fronteirapróximos ao limiar.

In [ ]:
y_arr = y_teste.valuesidx_reset = np.arange(len(X_teste))# Acertos de alta confiançavp_conf = idx_reset[(y_arr == 1) & (pred_teste == 1)]vp_conf = vp_conf[np.argsort(-prob_teste[vp_conf])][:2]vn_conf = idx_reset[(y_arr == 0) & (pred_teste == 0)]vn_conf = vn_conf[np.argsort(prob_teste[vn_conf])][:1]# Erros do modelofp = idx_reset[(y_arr == 0) & (pred_teste == 1)]fp = fp[np.argsort(-prob_teste[fp])][:1]fn = idx_reset[(y_arr == 1) & (pred_teste == 0)]fn = fn[np.argsort(prob_teste[fn])][:1]# Casos-fronteirafronteira = idx_reset[np.argsort(np.abs(prob_teste - limiar))][:2]selecionados = []for idx, tipo in ([(i, 'VP alta confiança') for i in vp_conf] +                  [(i, 'VN alta confiança') for i in vn_conf] +                  [(i, 'Falso Positivo') for i in fp] +                  [(i, 'Falso Negativo') for i in fn] +                  [(i, 'Caso-fronteira') for i in fronteira]):    if idx not in [s[0] for s in selecionados]:        selecionados.append((int(idx), tipo))tabela_casos = pd.DataFrame([    {'Índice': i, 'Tipo': t, 'Classe real': int(y_arr[i]),     'Classe prevista': int(pred_teste[i]),     'Probabilidade': round(float(prob_teste[i]), 4)}    for i, t in selecionados])display(tabela_casos)tabela_casos.to_csv(os.path.join(TAB_DIR, 'tab_casos_lime.csv'), index=False)

In [ ]:
explicador_lime = LimeTabularExplainer(    training_data=X_treino.values,    feature_names=COLUNAS,    class_names=['Não-evasor', 'Evasor'],    mode='classification',    discretize_continuous=True,    random_state=RANDOM_STATE)N_FEATURES = 10      # features reportadas por explicação (§3.5.2)N_AMOSTRAS = 5000    # exemplos perturbados por instância (§3.5.2)N_REPETICOES = 3     # média de 3 execuções — mitiga instabilidade (§2.7.3)print(f'Configuração: {N_FEATURES} features, {N_AMOSTRAS} perturbações, '      f'média de {N_REPETICOES} execuções')

In [ ]:
def explicar_lime(indice, n_rep=N_REPETICOES):    """Gera explicação LIME e retorna a média de n_rep execuções."""    acumulado = {}    for _ in range(n_rep):        exp = explicador_lime.explain_instance(            data_row=X_teste.iloc[indice].values,            predict_fn=modelo.predict_proba,            num_features=N_FEATURES,            num_samples=N_AMOSTRAS        )        for regra, peso in exp.as_list():            acumulado.setdefault(regra, []).append(peso)    media = {r: float(np.mean(p)) for r, p in acumulado.items()}    return (pd.DataFrame({'Regra': list(media.keys()),                          'Contribuição': list(media.values())})            .assign(abs_c=lambda d: d['Contribuição'].abs())            .sort_values('abs_c', ascending=False)            .drop(columns='abs_c')            .head(N_FEATURES)            .reset_index(drop=True))explicacoes = {}for indice, tipo in selecionados:    print(f'\n{"=" * 70}')    print(f'Instância {indice} — {tipo}')    print(f'Real: {"Evasor" if y_arr[indice] == 1 else "Não-evasor"} | '          f'Previsto: {"Evasor" if pred_teste[indice] == 1 else "Não-evasor"} | '          f'Probabilidade: {prob_teste[indice]:.4f}')    print('=' * 70)    df_exp = explicar_lime(indice)    df_exp['Direção'] = np.where(df_exp['Contribuição'] > 0,                                 'Aumenta risco', 'Reduz risco')    df_exp['Contribuição'] = df_exp['Contribuição'].round(4)    display(df_exp)    explicacoes[indice] = df_exp

In [ ]:
# Visualização das explicações locaisn = len(selecionados)fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(16, 4.2 * ((n + 1) // 2)))axes = np.array(axes).ravel()for ax, (indice, tipo) in zip(axes, selecionados):    d = explicacoes[indice].sort_values('Contribuição')    cores = ['#C73E1D' if v > 0 else '#2E86AB' for v in d['Contribuição']]    ax.barh(d['Regra'], d['Contribuição'], color=cores, alpha=0.85)    ax.axvline(0, color='#333333', linewidth=0.8)    ax.set_title(f'#{indice} — {tipo} (p = {prob_teste[indice]:.3f})', fontsize=11)    ax.set_xlabel('Contribuição para a predição')    ax.tick_params(axis='y', labelsize=8)for ax in axes[n:]:    ax.axis('off')plt.suptitle('Explicações locais LIME — vermelho aumenta risco, azul reduz', y=1.0)plt.tight_layout()salvar_fig('lime_explicacoes_locais')plt.show()

## 3. Síntese e tradução em insights de negócio (§3.5.3)Etapa final: consolidação dos achados globais (SHAP) e locais (LIME) em recomendaçõesacionáveis. **Este é o quadro que sustenta a entrega central do trabalho** — preencha acoluna de recomendações com base nos seus resultados reais.

In [ ]:
print('FATORES GLOBAIS DE MAIOR IMPACTO (SHAP)')print('=' * 70)for i, linha in importancia.head(8).iterrows():    print(f'{i + 1:>2}. {linha["Variável"]:<38} '          f'{linha["SHAP médio absoluto"]:>7.4f}  {linha["Direção predominante"]}')# Esqueleto para consolidação — preencha após analisar os resultadosrecomendacoes = pd.DataFrame([    {'Fator identificado': '', 'Evidência (SHAP/LIME)': '',     'Recomendação de negócio': '', 'Público-alvo': ''},])recomendacoes.to_csv(os.path.join(TAB_DIR, 'tab_recomendacoes_negocio.csv'),                     index=False)print(f'\nEsqueleto salvo em: tab_recomendacoes_negocio.csv')

In [ ]:
print('=' * 70)print('RESUMO DA ETAPA DE EXPLICABILIDADE')print('=' * 70)print(f'Modelo explicado: {modelo_nome} ({variante})')print(f'Instâncias com valores SHAP calculados: {len(X_teste):,}')print(f'Casos explicados individualmente via LIME: {len(selecionados)}')print(f'Figuras geradas em: {FIG_DIR}')print(f'Tabelas geradas em: {TAB_DIR}')print()print('Materiais prontos para o Capítulo 4 (Desenvolvimento) da monografia.')